# Connect to Google Drive

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
%load_ext rpy2.ipython


The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


# Read in file

In [10]:
%%R
library(data.table)

path <- "/content/drive/MyDrive/masters_thesis/NHANES/nhanes_1440_vssteps.csv"


steps <- fread(path, nrows = 1000000)

steps[1:10, 1:10]

     SEQN PAXDAYM PAXDAYWM min_0001 min_0002 min_0003 min_0004 min_0005
    <int>   <int>    <int>    <int>    <int>    <int>    <int>    <int>
 1: 73557       1        3       NA       NA       NA       NA       NA
 2: 73557       2        4        0        0        0        0        0
 3: 73557       3        5        0        0        0        0        0
 4: 73557       4        6        0        0        0        0        0
 5: 73557       5        7        0        0        0        0        0
 6: 73557       6        1        0        0        0        0        0
 7: 73557       7        2        0        0        0        0        0
 8: 73557       8        3        0        0        0        0        0
 9: 73557       9        4        0        0        0        0        0
10: 73558       1        2       NA       NA       NA       NA       NA
    min_0006 min_0007
       <int>    <int>
 1:       NA       NA
 2:        0        0
 3:        0        0
 4:        0        0
 5: 

data.table 1.18.4 using 1 threads (see ?getDTthreads).  Latest news: r-datatable.com

Attaching package: ‘data.table’

The following object is masked from ‘package:base’:

    %notin%



# Restrict to full days of data

In [11]:
%%R

steps <- steps[PAXDAYM %between% c(2, 8)]

In [12]:
%%R

steps[, PAXDAYWM := NULL]

steps[1:10, 1:10]

     SEQN PAXDAYM min_0001 min_0002 min_0003 min_0004 min_0005 min_0006
    <int>   <int>    <int>    <int>    <int>    <int>    <int>    <int>
 1: 73557       2        0        0        0        0        0        0
 2: 73557       3        0        0        0        0        0        0
 3: 73557       4        0        0        0        0        0        0
 4: 73557       5        0        0        0        0        0        0
 5: 73557       6        0        0        0        0        0        0
 6: 73557       7        0        0        0        0        0        0
 7: 73557       8        0        0        0        0        0        0
 8: 73558       2        4        0        0        0        0        0
 9: 73558       3        0        0        0        0        0        0
10: 73558       4        0        0        0        0        0        0
    min_0007 min_0008
       <int>    <int>
 1:        0        0
 2:        0        0
 3:        0        0
 4:        0        0
 5: 

# Make Transpose Table

In [13]:
%%R
library(data.table)

setDT(steps)

# get minute columns
min_cols <- grep("^min_", names(steps), value = TRUE)

# melt to long format
steps_long <- melt(
  steps,
  id.vars = c("SEQN", "PAXDAYM"),
  measure.vars = min_cols,
  variable.name = "MIN_of_day",
  value.name = "Steps_in_Min"
)

# convert minute label to numeric
steps_long[, MIN_of_day := as.numeric(gsub("min_", "", MIN_of_day))]

head(steps_long)

    SEQN PAXDAYM MIN_of_day Steps_in_Min
   <int>   <int>      <num>        <int>
1: 73557       2          1            0
2: 73557       3          1            0
3: 73557       4          1            0
4: 73557       5          1            0
5: 73557       6          1            0
6: 73557       7          1            0


# Add an hour column

In [14]:
%%R
library(data.table)

setDT(steps_long)

steps_long[, HOUR := (MIN_of_day - 1) %/% 60]

head(steps_long)

    SEQN PAXDAYM MIN_of_day Steps_in_Min  HOUR
   <int>   <int>      <num>        <int> <num>
1: 73557       2          1            0     0
2: 73557       3          1            0     0
3: 73557       4          1            0     0
4: 73557       5          1            0     0
5: 73557       6          1            0     0
6: 73557       7          1            0     0


# Make a percent minutes missing in hour column

In [ ]:
%%R
library(data.table)

setDT(steps_long)

steps_long[, pct_missing_in_hour :=
  100 * sum(is.na(Steps_in_Min)) / .N,
  by = .(SEQN, PAXDAYM, HOUR)
]

head(steps_long)

    SEQN PAXDAYM MIN_of_day Steps_in_Min  HOUR pct_missing_in_hour
   <int>   <int>      <num>        <int> <num>               <num>
1: 73557       2          1            0     0                   0
2: 73557       3          1            0     0                   0
3: 73557       4          1            0     0                   0
4: 73557       5          1            0     0                   0
5: 73557       6          1            0     0                   0
6: 73557       7          1            0     0                   0


# Get one patient and check

In [ ]:
%%R
library(data.table)
library(ggplot2)

setDT(steps_long)

# pick one patient
one_id <- steps_long$SEQN[1]

# subset one patient
pt <- steps_long[SEQN == one_id]

# if you want just one day too, take the first day
one_day <- pt$PAXDAYM[1]
pt_day <- pt[PAXDAYM == one_day]

# hour-level validation table
check_hour <- pt_day[
  , .(
      n_minutes = .N,
      n_missing = sum(is.na(Steps_in_Min)),
      pct_missing_calc = 100 * sum(is.na(Steps_in_Min)) / .N,
      pct_missing_column = unique(pct_missing_in_hour)
    ),
  by = .(HOUR)
][order(HOUR)]

print(check_hour)

     HOUR n_minutes n_missing pct_missing_calc pct_missing_column
    <num>     <int>     <int>            <num>              <num>
 1:     0        60         0                0                  0
 2:     1        60         0                0                  0
 3:     2        60         0                0                  0
 4:     3        60         0                0                  0
 5:     4        60         0                0                  0
 6:     5        60         0                0                  0
 7:     6        60         0                0                  0
 8:     7        60         0                0                  0
 9:     8        60         0                0                  0
10:     9        60         0                0                  0
11:    10        60         0                0                  0
12:    11        60         0                0                  0
13:    12        60         0                0                  0
14:    13 

In [ ]:
%%R
check_hour[, matches := all.equal(pct_missing_calc, pct_missing_column), by = HOUR]
print(check_hour)

     HOUR n_minutes n_missing pct_missing_calc pct_missing_column matches
    <num>     <int>     <int>            <num>              <num>  <lgcl>
 1:     0        60         0                0                  0    TRUE
 2:     1        60         0                0                  0    TRUE
 3:     2        60         0                0                  0    TRUE
 4:     3        60         0                0                  0    TRUE
 5:     4        60         0                0                  0    TRUE
 6:     5        60         0                0                  0    TRUE
 7:     6        60         0                0                  0    TRUE
 8:     7        60         0                0                  0    TRUE
 9:     8        60         0                0                  0    TRUE
10:     9        60         0                0                  0    TRUE
11:    10        60         0                0                  0    TRUE
12:    11        60         0         

# Remove all tables except Steps_long

In [ ]:
%%R
rm(list = setdiff(ls(), "steps_long"))
gc()

            used   (Mb) gc trigger    (Mb)  max used   (Mb)
Ncells   1039134   55.5    1888410   100.9   1888410  100.9
Vcells 731531885 5581.2 1367743256 10435.1 951315674 7258.0


In [ ]:
%%R
gc()

            used   (Mb) gc trigger    (Mb)  max used   (Mb)
Ncells   1039376   55.6    1888410   100.9   1888410  100.9
Vcells 731532559 5581.2 1367743256 10435.1 951315674 7258.0


In [ ]:
%%R
ls()

[1] "steps_long"


# Get counts for 20 hour eligibility


In [ ]:
%%R
library(data.table)

setDT(steps_long)

# one row per person-day-hour
hour_check <- steps_long[
  , .(pct_missing_in_hour = pct_missing_in_hour[1]),
  by = .(SEQN, PAXDAYM, HOUR)
]

# for each person-day, count valid hours
day_check <- hour_check[
  , .(
      n_valid_hours = sum(pct_missing_in_hour <= 10, na.rm = TRUE)
    ),
  by = .(SEQN, PAXDAYM)
]

# mark whether each day passes the 20-hour rule
day_check[, day_pass := n_valid_hours >= 20]

# for each SEQN, require all available days to pass
seqn_check <- day_check[
  , .(
      n_days = .N,
      n_days_pass = sum(day_pass),
      seqn_pass = all(day_pass)
    ),
  by = SEQN
]

# counts and percentage
n_seqn_total <- uniqueN(seqn_check$SEQN)
n_seqn_pass <- seqn_check[, sum(seqn_pass)]
pct_seqn_pass <- 100 * n_seqn_pass / n_seqn_total

cat("Number of SEQNs meeting the rule:", n_seqn_pass, "\n")
cat("Total number of SEQNs:", n_seqn_total, "\n")
cat("Percent of SEQNs meeting the rule:", round(pct_seqn_pass, 2), "%\n")

# Collapse hours

In [ ]:
%%R
library(data.table)

setDT(steps_long)

steps_hour <- steps_long[
  ,
  .(steps_in_hour = sum(Steps_in_Min, na.rm = TRUE)),
  by = .(SEQN, PAXDAYM, HOUR)
]

head(steps_hour)

    SEQN PAXDAYM  HOUR steps_in_hour
   <int>   <int> <num>         <int>
1: 73557       2     0             0
2: 73557       3     0             0
3: 73557       4     0             0
4: 73557       5     0             0
5: 73557       6     0             0
6: 73557       7     0             0


In [ ]:
%%R
library(data.table)

setDT(steps_long)

steps_hour <- steps_long[
  ,
  .(steps_in_hour = sum(Steps_in_Min, na.rm = TRUE)),
  by = .(SEQN, PAXDAYM, HOUR)
]

head(steps_hour)

In [ ]:
%%R

head(steps_hour,100)

In [ ]:
%%R
dim(steps_hour)

# Make Plots

In [ ]:
%%R

library(dplyr)
library(ggplot2)

# choose a patient and a day
patient_id <- 73558
day_id <- 2

# filter data
plot_data <- steps_hour %>%
  filter(SEQN == patient_id, PAXDAYM == day_id)

# plot
ggplot(plot_data, aes(x = HOUR, y = steps_in_hour)) +
  geom_line(size = 1) +
  geom_point(size = 2) +
  labs(
    title = paste("Hourly Steps for Patient", patient_id, "Day", day_id),
    x = "Hour of Day",
    y = "Steps in Hour"
  ) +
  theme_minimal()

In [ ]:
%%R

library(dplyr)
library(ggplot2)

# choose a participant
patient_id <- 73558

# filter to that participant
plot_data <- steps_hour %>%
  filter(SEQN == patient_id)

# create continuous time across the week
plot_data <- plot_data %>%
  mutate(hour_of_week = (PAXDAYM - 1) * 24 + HOUR)

# plot
ggplot(plot_data, aes(x = hour_of_week, y = steps_in_hour)) +
  geom_line(size = 1) +
  geom_point(size = 1.5) +
  labs(
    title = paste("Weekly Steps Pattern for Patient", patient_id),
    x = "Hour of Week",
    y = "Steps per Hour"
  ) +
  theme_minimal()

# Save Steps_hour

In [ ]:
%%R
write.csv(steps_hour, "steps_hour.csv", row.names = FALSE)

In [ ]:
%%R
write.csv(steps_hour, "/content/drive/MyDrive/masters_thesis/NHANES-2-Cleaner/steps_hour.csv", row.names = FALSE)